# Reward Modeling — Bangla নোটবুক

এটি `example.py`-এর একটি Bangla-অনুবাদিত, চালানোযোগ্য নোটবুক সংস্করণ।

Reward Modeling

PyTorch-এ scratch থেকে Bradley-Terry pairwise preference loss implement
করে এবং এটি দিয়ে একটি ছোট reward model (RM)-কে SYNTHETIC preference
data-তে প্রশিক্ষণ দেয় -- যেটি একটি পরিচিত ground-truth scoring function
থেকে উৎপন্ন -- যাতে সরাসরি সঠিকতা যাচাই করা যায়, যা বাস্তব মানুষের label
দিয়ে অসম্ভব। এখানে "response" হলো খেলনা feature vector (যেমন factual
accuracy, relevance, politeness, যথাযথ দৈর্ঘ্য -- এমন বৈশিষ্ট্যের
প্রতিনিধি); সেই feature-গুলির একটি নির্দিষ্ট, লুকানো linear function
হলো ground truth "true quality"। মানুষের preference label সিমুলেট করা
হয় সত্যিকারের স্কোরের উপর Bradley-Terry distribution থেকে SAMPLING
করে (সবসময় উচ্চ-স্কোরিংটি বেছে নেওয়া নয়), যাতে বাস্তবসম্মত, মাঝে
মাঝে noisy মানুষের বিচার অনুকরণ করা যায়। RM-কে শুধুমাত্র এই pairwise
comparison-এ (সে কখনো সরাসরি true score দেখে না) প্রশিক্ষণ দেওয়ার পর
পরীক্ষা করি তার শেখা স্কোর held-out items-এ লুকানো true ground truth-এর
সাথে দৃঢ়ভাবে সম্পর্কযুক্ত কিনা।

Runtime: CPU-তে কয়েক সেকেন্ড।

চালান:
    নোটবুকের সব code cell উপরে থেকে নিচে চালান (Cell -> Run All)।

## কীভাবে চালাবেন

উপর থেকে নিচে (Cell → Run All) সব cell চালান। Runtime: CPU-তে কয়েক সেকেন্ড।

In [ ]:
"""Reward Modeling

PyTorch-এ scratch থেকে Bradley-Terry pairwise preference loss implement
করে এবং এটি দিয়ে একটি ছোট reward model (RM)-কে SYNTHETIC preference
data-তে প্রশিক্ষণ দেয় -- যেটি একটি পরিচিত ground-truth scoring function
থেকে উৎপন্ন -- যাতে সরাসরি সঠিকতা যাচাই করা যায়, যা বাস্তব মানুষের label
দিয়ে অসম্ভব। এখানে "response" হলো খেলনা feature vector (যেমন factual
accuracy, relevance, politeness, যথাযথ দৈর্ঘ্য -- এমন বৈশিষ্ট্যের
প্রতিনিধি); সেই feature-গুলির একটি নির্দিষ্ট, লুকানো linear function
হলো ground truth "true quality"। মানুষের preference label সিমুলেট করা
হয় সত্যিকারের স্কোরের উপর Bradley-Terry distribution থেকে SAMPLING
করে (সবসময় উচ্চ-স্কোরিংটি বেছে নেওয়া নয়), যাতে বাস্তবসম্মত, মাঝে
মাঝে noisy মানুষের বিচার অনুকরণ করা যায়। RM-কে শুধুমাত্র এই pairwise
comparison-এ (সে কখনো সরাসরি true score দেখে না) প্রশিক্ষণ দেওয়ার পর
পরীক্ষা করি তার শেখা স্কোর held-out items-এ লুকানো true ground truth-এর
সাথে দৃঢ়ভাবে সম্পর্কযুক্ত কিনা।

Runtime: CPU-তে কয়েক সেকেন্ড।

চালান:
    নোটবুকের সব code cell উপরে থেকে নিচে চালান (Cell -> Run All)।
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

## 0. পরিচিত ground-truth scorer থেকে SYNTHETIC preference data

"Response"-গুলো হলো 8-মাত্রিক feature vector; গোপন ground-truth reward হলো সেই feature-গুলির একটি নির্দিষ্ট রৈখিক (linear) ওজন। নিচের ফাংশনগুলো সেই ground truth থেকে preference জোড়া তৈরি করে।

In [ ]:
FEATURE_DIM = 8   # একটি খেলনা "response" হলো একটি 8-মাত্রিক feature vector

# Ground-truth reward function: feature-গুলির একটি নির্দিষ্ট, লুকানো linear
# weighting (যেমন w[0] হতে পারে "কতটা factual accuracy", w[1] "কতটা polite",
# ইত্যাদি -- RM এই weight-গুলো কখনো দেখতে পাবে না, কেবল ফলে আসা স্কোরের
# pairwise comparison-ই দেখবে)।
TRUE_WEIGHTS = torch.tensor([1.8, -1.2, 0.9, 0.5, -0.7, 1.1, -0.3, 0.6])


def true_reward(features):
    """লুকানো ground-truth scorer। Reward model এটিকে কখনো দেখে না --
    এটি শুধু এখানে (a) preference label তৈরি করতে এবং (b) পরে RM-এর
    শেখা ranking গ্রেড করতে ব্যবহৃত হয়।"""
    return features @ TRUE_WEIGHTS


def make_items(n):
    return torch.rand(n, FEATURE_DIM) * 2 - 1   # feature গুলো [-1, 1]-এ


def make_preference_pairs(items, num_pairs, label_noise=True):
    """আইটেমের জোড়া randomভাবে নমুনা করুন এবং প্রতিটির জন্য একটি
    preference label তৈরি করুন।

    label_noise=True হলে, "মানুষের" পছন্দটি Bradley-Terry probability
    P(i, j-এর উপর পছন্দ) = sigmoid(true_r_i - true_r_j) থেকে SAMPLED হয় --
    হুবহু বাস্তব মানুষ labeler-দের মতো যারা পুরোপুরি সামঞ্জস্যপূর্ণ নয় --
    নির্ধারকভাবে সবসময় উচ্চ-সত্য-স্কোর আইটেমটি বেছে নেওয়ার বদলে। এটি
    বাস্তব preference data-র আরও সৎ সিমুলেশন।
    """
    n = items.shape[0]
    idx_a = torch.randint(0, n, (num_pairs,))
    idx_b = torch.randint(0, n, (num_pairs,))
    true_scores = true_reward(items)

    r_a, r_b = true_scores[idx_a], true_scores[idx_b]
    if label_noise:
        prob_a_wins = torch.sigmoid(r_a - r_b)
        a_wins = torch.bernoulli(prob_a_wins).bool()
    else:
        a_wins = (r_a > r_b)

    # (chosen_idx, rejected_idx) আকারে সাজান
    chosen = torch.where(a_wins, idx_a, idx_b)
    rejected = torch.where(a_wins, idx_b, idx_a)
    return chosen, rejected

## 1. Reward model: একটি ছোট MLP scalar head

"pretrained backbone + scalar head"-এর (Lesson README section 4) পরিবর্ত হিসেবে কাজ করে — pooled feature কী দিয়ে তৈরি হয় তা নির্বিশেষে Bradley-Terry প্রক্রিয়া অভিন্ন।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Reward model: একটি ছোট MLP scalar head -- "pretrained backbone +
# scalar head"-এর পরিবর্ত (Lesson README section 4) -- pooled feature
# যা কিছুই তৈরি করুক না কেন Bradley-Terry প্রক্রিয়া অভিন্ন।
# ---------------------------------------------------------------------------

class RewardModel(nn.Module):
    def __init__(self, feature_dim, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)   # (batch,) প্রতিটি আইটেমের scalar reward


def bradley_terry_loss(reward_model, items, chosen_idx, rejected_idx):
    """loss = -log( sigmoid( r(chosen) - r(rejected) ) ), batch জুড়ে গড়।
    এটি হুবহু Bradley-Terry মডেলের অধীনে পর্যবেক্ষিত মানুষের পছন্দের
    negative log-likelihood (Lesson README section 3)।"""
    r_chosen = reward_model(items[chosen_idx])
    r_rejected = reward_model(items[rejected_idx])
    return -F.logsigmoid(r_chosen - r_rejected).mean()

## 2. মূল্যায়ন utility: শেখা RM-কে পরিচিত ground truth-এর বিরুদ্ধে যাচাই

বাস্তব RLHF পাইপলাইন যা কখনো সরাসরি করতে পারে না — এখানে তা সম্ভব, কারণ ground truth আমাদের জানা।

In [ ]:
# ---------------------------------------------------------------------------
# 2. মূল্যায়ন utility: শেখা RM-কে পরিচিত ground truth-এর বিরুদ্ধে যাচাই
# করুন -- বাস্তব RLHF পাইপলাইন যা কখনো সরাসরি করতে পারে না।
# ---------------------------------------------------------------------------

def spearman_correlation(a, b):
    """ম্যানুয়াল Spearman rank correlation (কোনো scipy dependency নেই):
    দুইটি স্কোর vector-এর কাঁচা মান নয়, তাদের RANK-কে সম্পর্কযুক্ত করে --
    যা আমরা ঠিক চাই, কারণ RM-এর কাঁচা স্কেল অর্থহীন (README section 2) --
    শুধুমাত্র এটি যে ORDERING induces করে তা ground truth-এর সাথে মিলতে হবে।"""
    def rank(x):
        order = np.argsort(x)
        ranks = np.empty_like(order, dtype=float)
        ranks[order] = np.arange(len(x))
        return ranks

    ra, rb = rank(a), rank(b)
    return np.corrcoef(ra, rb)[0, 1]


def pairwise_accuracy(reward_model, items, true_scores, num_test_pairs=2000):
    """Random held-out জোড়া নমুনা করুন এবং দেখুন RM-এর PREFERENCE (যেটিকে
    সে বেশি স্কোর করে) ground truth-এর পছন্দের সাথে কতবার মেলে -- "সে ঠিক
    জিনিসটি পছন্দ করতে শিখেছে কিনা"-র সবচেয়ে সরাসরি পরিমাপ, যেকোনো কাঁচা
    স্কোর স্কেল বা offset-এর থেকে স্বাধীন।"""
    n = items.shape[0]
    idx_a = torch.randint(0, n, (num_test_pairs,))
    idx_b = torch.randint(0, n, (num_test_pairs,))
    with torch.no_grad():
        rm_scores = reward_model(items)
    rm_a, rm_b = rm_scores[idx_a], rm_scores[idx_b]
    true_a, true_b = true_scores[idx_a], true_scores[idx_b]

    rm_prefers_a = rm_a > rm_b
    true_prefers_a = true_a > true_b
    agree = (rm_prefers_a == true_prefers_a).float().mean().item()
    return agree

## সম্পূর্ণ ডেমো চালানো

`main()` তিনটি ধাপ চালায়: পরিচিত ground truth থেকে synthetic preference জোড়া তৈরি, Bradley-Terry loss দিয়ে RM-কে প্রশিক্ষণ, এবং held-out ডেটায় RM-এর শেখা ranking-কে লুকানো ground truth-এর বিরুদ্ধে যাচাই।

In [ ]:
def main():
    print("=" * 70)
    print("1. SYNTHETIC PREFERENCE DATA FROM A KNOWN GROUND-TRUTH SCORER")
    print("=" * 70)
    print(f"Ground truth is a hidden linear function of {FEATURE_DIM} toy features")
    print(f"(true weights, NEVER shown to the reward model): "
          f"{[round(w, 2) for w in TRUE_WEIGHTS.tolist()]}")

    train_items = make_items(400)
    test_items = make_items(300)
    test_true_scores = true_reward(test_items)

    NUM_PAIRS = 3000
    chosen_idx, rejected_idx = make_preference_pairs(train_items, NUM_PAIRS, label_noise=True)

    # Sanity check: "noisy human" labeler কতবার ground truth-র নিজস্ব
    # ranking-এর সাথে আসলে মেলে? বাস্তব মানুষ labeler-ও পুরোপুরি সামঞ্জস্যপূর্ণ
    # নয় -- এটি দেখায় আমাদের সিমুলেশন তুচ্ছ নয়।
    true_scores_train = true_reward(train_items)
    label_matches_ground_truth = (
        true_scores_train[chosen_idx] > true_scores_train[rejected_idx]
    ).float().mean().item()
    print(f"\nGenerated {NUM_PAIRS} pairwise comparisons with simulated (noisy) human labels.")
    print(f"Fraction of labels that agree with the ground-truth ranking: "
          f"{label_matches_ground_truth:.3f}")
    print("-> Not 1.000 -- some comparisons are close calls where the simulated")
    print("   'human' picks the slightly-worse item, exactly like real labelers")
    print("   disagreeing on ambiguous cases (README section 5).")

    print("\n" + "=" * 70)
    print("2. TRAINING THE REWARD MODEL WITH THE BRADLEY-TERRY LOSS")
    print("=" * 70)
    print("loss = -log( sigmoid( r(chosen) - r(rejected) ) )")
    print("The RM NEVER sees a true score directly -- only which item won each pair.\n")

    reward_model = RewardModel(FEATURE_DIM)
    optimizer = torch.optim.Adam(reward_model.parameters(), lr=1e-2)

    BATCH_SIZE = 64
    NUM_STEPS = 800
    for step in range(1, NUM_STEPS + 1):
        batch = torch.randint(0, NUM_PAIRS, (BATCH_SIZE,))
        loss = bradley_terry_loss(
            reward_model, train_items, chosen_idx[batch], rejected_idx[batch]
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 100 == 0 or step == 1:
            print(f"  step {step:4d}   bradley-terry loss = {loss.item():.4f}")

    print("\n" + "=" * 70)
    print("3. VERIFYING THE LEARNED REWARD MODEL AGAINST THE HIDDEN GROUND TRUTH")
    print("=" * 70)
    with torch.no_grad():
        learned_scores = reward_model(test_items).numpy()
    true_scores_np = test_true_scores.numpy()

    corr = spearman_correlation(learned_scores, true_scores_np)
    acc = pairwise_accuracy(reward_model, test_items, test_true_scores)

    print(f"Spearman rank correlation between learned RM scores and the true")
    print(f"hidden ground-truth scores, on {test_items.shape[0]} held-out items: {corr:.3f}")
    print(f"Pairwise agreement between the RM's preferences and the ground")
    print(f"truth's preferences, over 2000 random held-out pairs: {acc:.3f}")

    print(f"\nFor comparison, the noisy simulated 'human' labels themselves only")
    print(f"agreed with ground truth {label_matches_ground_truth:.3f} of the time on")
    print(f"TRAINING pairs -- yet the trained RM's pairwise agreement with ground")
    print(f"truth on held-out data is {acc:.3f}. Averaging the Bradley-Terry loss over")
    print(f"thousands of noisy comparisons recovers a scorer that is MORE accurate")
    print(f"than any single noisy label it was trained on -- exactly the effect that")
    print(f"makes reward modeling from imperfect human preferences work in practice.")

    print("\nSample of learned vs. true scores on 5 held-out items (raw values --")
    print("remember from README section 2 that only relative order is meaningful,")
    print("not the absolute scale, so the numbers themselves needn't match):")
    print(f"{'true score':>14}{'learned score':>16}")
    for i in range(5):
        print(f"{true_scores_np[i]:>14.3f}{learned_scores[i]:>16.3f}")

main()